# 01 Raw Data Discovery

Purpose: inspect `data/raw/matches_raw.csv` before Day 3 cleaning.

This notebook does **not** create the canonical cleaned dataset.  
It is exploration only.

The goal is to answer:

1. What columns are available?
2. Which columns should become canonical system inputs?
3. Which leagues/seasons are present?
4. Which date/result/odds columns are reliable?
5. What issues must Day 3 cleaning handle?
6. What tests and schema rules should be added to the system?

Expected input:

```text
data/raw/matches_raw.csv
```

Expected optional Day 2 references:

```text
outputs/evaluation/raw_schema_report.csv
outputs/evaluation/selected_external_files.csv
```

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "matches_raw.csv"
SCHEMA_REPORT_PATH = PROJECT_ROOT / "outputs" / "evaluation" / "raw_schema_report.csv"
SELECTED_FILES_PATH = PROJECT_ROOT / "outputs" / "evaluation" / "selected_external_files.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw path exists: {RAW_PATH.exists()} -> {RAW_PATH}")
print(f"Schema report exists: {SCHEMA_REPORT_PATH.exists()} -> {SCHEMA_REPORT_PATH}")
print(f"Selected files exists: {SELECTED_FILES_PATH.exists()} -> {SELECTED_FILES_PATH}")

Project root: c:\Users\jaymi\Projects\Continuous Projects\football-decision-system
Raw path exists: True -> c:\Users\jaymi\Projects\Continuous Projects\football-decision-system\data\raw\matches_raw.csv
Schema report exists: True -> c:\Users\jaymi\Projects\Continuous Projects\football-decision-system\outputs\evaluation\raw_schema_report.csv
Selected files exists: True -> c:\Users\jaymi\Projects\Continuous Projects\football-decision-system\outputs\evaluation\selected_external_files.csv


## 1. Load raw data

We load with `low_memory=False` to avoid pandas guessing column types in chunks and yelling about mixed dtypes.  
This is **not cleaning**. It just makes inspection less chaotic.

In [2]:
df = pd.read_csv(RAW_PATH, low_memory=False)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

df.head()

Rows: 20,296
Columns: 179


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,BWH,BWD,BWA,IWH,IWD,IWA,PSH,PSD,PSA,WHH,WHD,WHA,VCH,VCD,VCA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,B365>2.5,B365<2.5,P>2.5,P<2.5,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,AHh,B365AHH,B365AHA,PAHH,PAHA,MaxAHH,MaxAHA,AvgAHH,AvgAHA,B365CH,B365CD,B365CA,BWCH,BWCD,BWCA,IWCH,IWCD,IWCA,PSCH,PSCD,PSCA,WHCH,WHCD,WHCA,VCCH,VCCD,VCCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,B365C>2.5,B365C<2.5,PC>2.5,PC<2.5,MaxC>2.5,MaxC<2.5,AvgC>2.5,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,source_country,source_season,source_season_start_year,source_league_code,source_file,source_layout,Unnamed: 105,BFH,BFD,BFA,1XBH,1XBD,1XBA,BFEH,BFED,BFEA,BFE>2.5,BFE<2.5,BFEAHH,BFEAHA,BFCH,BFCD,BFCA,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA,Unnamed: 119,Unnamed: 120,BFDH,BFDD,BFDA,BMGMH,BMGMD,BMGMA,BVH,BVD,BVA,CLH,CLD,CLA,LBH,LBD,LBA,BFDCH,BFDCD,BFDCA,BMGMCH,BMGMCD,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA,Referee,Country,League,Season,Home,Away,HG,AG,Res
0,B1,08/08/2020,15:30,Club Brugge,Charleroi,0.0,1.0,A,0.0,0.0,D,17.0,6.0,5.0,4.0,10.0,11.0,10.0,2.0,2.0,3.0,0.0,0.0,1.50,3.8,6.00,1.5,4.00,6.50,1.53,3.80,6.00,1.53,4.13,6.81,1.50,4.0,6.50,1.53,4.0,6.5,1.59,4.33,7.10,1.52,4.00,6.26,1.95,1.85,1.93,1.90,2.00,1.99,1.91,1.87,-1.00,1.95,1.90,1.96,1.88,2.02,1.97,1.94,1.89,1.57,3.75,5.25,1.62,3.80,5.75,1.63,3.60,5.50,1.63,3.97,5.75,1.57,3.80,6.00,1.62,3.90,5.75,1.67,4.05,6.45,1.61,3.81,5.63,2.05,1.75,2.05,1.83,2.13,1.88,2.03,1.78,-0.75,1.77,2.10,1.82,2.09,1.85,2.12,1.79,2.06,belgium,2021,2020.0,B1,data\external\belgium\2021\B1.csv,country_season_file,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B1,08/08/2020,18:00,Antwerp,Mouscron,1.0,1.0,D,0.0,0.0,D,12.0,4.0,9.0,2.0,8.0,5.0,7.0,3.0,2.0,1.0,0.0,0.0,1.30,5.0,10.00,1.3,5.25,9.00,1.33,4.80,8.75,1.32,5.34,10.09,1.29,5.0,11.00,NaN,NaN,NaN,1.37,5.50,11.50,1.31,5.11,9.27,1.66,2.15,1.72,2.17,1.78,2.27,1.68,2.14,-1.50,2.00,1.85,2.09,1.79,2.12,1.90,1.99,1.84,1.44,4.20,6.00,1.48,4.60,6.25,1.47,4.10,6.25,1.52,4.41,6.35,1.44,4.50,6.50,1.45,4.60,7.00,1.52,5.13,7.70,1.46,4.42,6.52,1.70,2.10,1.73,2.19,1.78,2.27,1.69,2.13,-1.00,1.83,2.02,1.89,2.01,1.90,2.25,1.79,2.05,belgium,2021,2020.0,B1,data\external\belgium\2021\B1.csv,country_season_file,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B1,08/08/2020,18:00,Standard,Cercle Brugge,1.0,0.0,H,0.0,0.0,D,11.0,6.0,8.0,1.0,10.0,11.0,5.0,4.0,1.0,1.0,0.0,0.0,1.40,4.5,6.50,1.4,4.75,6.75,1.43,4.40,6.75,1.43,4.75,7.44,1.40,4.5,7.50,1.44,4.5,7.0,1.50,4.98,7.50,1.43,4.56,6.84,1.70,2.10,1.72,2.18,1.78,2.28,1.69,2.14,-1.25,2.00,1.85,2.03,1.82,2.06,1.90,1.98,1.85,1.45,4.50,6.50,1.48,4.40,6.75,1.45,4.10,6.75,1.51,4.16,7.25,1.44,4.33,7.00,1.50,4.33,6.50,1.54,4.65,7.50,1.48,4.26,6.60,1.72,2.07,1.77,2.12,1.83,2.19,1.73,2.09,-1.00,1.85,2.00,1.91,1.99,1.97,2.13,1.86,1.99,belgium,2021,2020.0,B1,data\external\belgium\2021\B1.csv,country_season_file,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B1,09/08/2020,12:30,St Truiden,Gent,2.0,1.0,H,1.0,1.0,D,11.0,13.0,8.0,2.0,19.0,13.0,6.0,3.0,2.0,2.0,0.0,0.0,4.50,3.8,1.65,4.6,4.00,1.67,4.30,3.85,1.70,4.54,4.19,1.72,4.33,4.0,1.70,NaN,NaN,NaN,4.70,4.50,1.76,4.36,4.05,1.70,1.53,2.40,1.56,2.47,1.57,2.53,1.54,2.43,0.75,1.93,1.93,1.96,1.88,1.97,1

## 2. Basic structure

In [3]:
basic_summary = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "duplicate_full_rows",
            "memory_usage_mb",
        ],
        "value": [
            len(df),
            len(df.columns),
            int(df.duplicated().sum()),
            round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        ],
    }
)

basic_summary

,metric,value
0,row_count,20296.0
1,column_count,179.0
2,duplicate_full_rows,0.0
3,memory_usage_mb,29.8


In [4]:
columns_df = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "non_null": [int(df[col].notna().sum()) for col in df.columns],
        "null_count": [int(df[col].isna().sum()) for col in df.columns],
        "null_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
        "unique_count": [int(df[col].nunique(dropna=True)) for col in df.columns],
    }
).sort_values(["null_pct", "column"], ascending=[False, True])

columns_df

,column,dtype,non_null,null_count,null_pct,unique_count
111,Unnamed: 105,float64,0,20296,100.00,0
138,Unnamed: 119,float64,0,20296,100.00,0
139,Unnamed: 120,float64,0,20296,100.00,0
166,CLCA,float64,1598,18698,92.13,87
165,CLCD,float64,1598,18698,92.13,40
164,CLCH,float64,1598,18698,92.13,91
169,LBCA,float64,1598,18698,92.13,87
168,LBCD,float64,1598,18698,92.13,39
167,LBCH,float64,1598,18698,92.13,90
151,CLA,float64,1640,18656,91.92,87


## 3. Source metadata coverage

These columns were added during Day 2 to preserve lineage.  
If they are missing, stop and fix Day 2 before cleaning.

In [5]:
source_cols = [
    "source_country",
    "source_season",
    "source_season_start_year",
    "source_league_code",
    "source_file",
    "source_layout",
]

missing_source_cols = [col for col in source_cols if col not in df.columns]
print("Missing source metadata columns:", missing_source_cols)

if not missing_source_cols:
    source_coverage = (
        df.groupby(["source_country", "source_league_code", "source_season", "source_layout"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["source_country", "source_league_code", "source_season"])
    )
    display(source_coverage)
else:
    print("Source coverage skipped because metadata columns are missing.")

Missing source metadata columns: []


,source_country,source_league_code,source_season,source_layout,rows
0,belgium,B1,2021,country_season_file,306
1,belgium,B1,2122,country_season_file,306
2,belgium,B1,2223,country_season_file,306
3,belgium,B1,2324,country_season_file,312
4,belgium,B1,2425,country_season_file,312
5,belgium,B1,2526,country_season_file,240
6,england,E0,2021,country_season_file,380
7,england,E0,2122,country_season_file,380
8,england,E0,2223,country_season_file,380
9,england,E0,2324,country_season_file,380


In [6]:
if "source_country" in df.columns:
    display(
        df["source_country"]
        .value_counts(dropna=False)
        .rename_axis("source_country")
        .reset_index(name="rows")
    )

,source_country,rows
0,mexico,4595
1,england,2209
2,italy,2200
3,spain,2190
4,france,1994
5,belgium,1782
6,netherlands,1782
7,germany,1773
8,portugal,1771


## 4. Candidate canonical columns

Day 3 needs a canonical match dataset.  
This section checks which raw columns look like candidates.

Target canonical columns we likely need:

```text
match_date
season
country
league
home_team
away_team
home_goals
away_goals
full_time_result
home_odds
draw_odds
away_odds
source_country
source_season
source_league_code
source_file
```

We are not renaming yet. We are mapping possibilities.

In [7]:
canonical_candidates = {
    "match_date": ["Date", "date", "MatchDate", "match_date"],
    "season": ["Season", "season", "source_season", "source_season_start_year"],
    "country": ["Country", "country", "source_country"],
    "league": ["League", "league", "Div", "source_league_code"],
    "home_team": ["HomeTeam", "Home", "home_team"],
    "away_team": ["AwayTeam", "Away", "away_team"],
    "home_goals": ["FTHG", "HG", "HomeGoals", "home_goals"],
    "away_goals": ["FTAG", "AG", "AwayGoals", "away_goals"],
    "full_time_result": ["FTR", "Res", "Result", "full_time_result"],
    "home_odds": ["B365H", "AvgH", "MaxH", "PSH", "WHH", "IWH", "home_odds"],
    "draw_odds": ["B365D", "AvgD", "MaxD", "PSD", "WHD", "IWD", "draw_odds"],
    "away_odds": ["B365A", "AvgA", "MaxA", "PSA", "WHA", "IWA", "away_odds"],
}

available_mapping = []

for canonical_name, candidates in canonical_candidates.items():
    found = [col for col in candidates if col in df.columns]
    available_mapping.append(
        {
            "canonical_name": canonical_name,
            "candidate_columns": candidates,
            "found_columns": found,
            "status": "found" if found else "missing",
        }
    )

mapping_df = pd.DataFrame(available_mapping)
mapping_df

,canonical_name,candidate_columns,found_columns,status
0,match_date,"[Date, date, MatchDate, match_date]",[Date],found
1,season,"[Season, season, source_season, source_season_start_year]","[Season, source_season, source_season_start_year]",found
2,country,"[Country, country, source_country]","[Country, source_country]",found
3,league,"[League, league, Div, source_league_code]","[League, Div, source_league_code]",found
4,home_team,"[HomeTeam, Home, home_team]","[HomeTeam, Home]",found
5,away_team,"[AwayTeam, Away, away_team]","[AwayTeam, Away]",found
6,home_goals,"[FTHG, HG, HomeGoals, home_goals]","[FTHG, HG]",found
7,away_goals,"[FTAG, AG, AwayGoals, away_goals]","[FTAG, AG]",found
8,full_time_result,"[FTR, Res, Result, full_time_result]","[FTR, Res]",found
9,home_odds,"[B365H, AvgH, MaxH, PSH, WHH, IWH, home_odds]","[B365H, AvgH, MaxH, PSH, WHH, IWH]",found


## 5. Date column inspection

Day 3 will need a reliable `match_date`.

This checks likely date columns and whether pandas can parse them.  
No parsed date is saved back to raw data here.

In [8]:
possible_date_cols = [col for col in df.columns if "date" in col.lower()] + [col for col in ["Date"] if col in df.columns]
possible_date_cols = sorted(set(possible_date_cols))

date_parse_records = []

for col in possible_date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    date_parse_records.append(
        {
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": int(df[col].notna().sum()),
            "parse_success_count": int(parsed.notna().sum()),
            "parse_success_pct": round(parsed.notna().mean() * 100, 2),
            "min_date": parsed.min(),
            "max_date": parsed.max(),
            "sample_values": df[col].dropna().astype(str).drop_duplicates().head(10).tolist(),
        }
    )

date_parse_df = pd.DataFrame(date_parse_records)
date_parse_df

,column,dtype,non_null,parse_success_count,parse_success_pct,min_date,max_date,sample_values
0,Date,str,20296,20296,100.0,2012-07-21,2026-03-23,"[08/08/2020, 09/08/2020, 10/08/2020, 14/08/2020, 15/08/2020, 16/08/2020, 17/08/2020, 21/08/2020, 22/08/2020, 23/08/2..."


In [9]:
if "Date" in df.columns:
    parsed_date = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    date_year_counts = (
        parsed_date.dt.year.value_counts(dropna=False)
        .sort_index()
        .rename_axis("match_year")
        .reset_index(name="rows")
    )
    display(date_year_counts)
else:
    print("No raw Date column found.")

,match_year,rows
0,2012,167
1,2013,334
2,2014,334
3,2015,334
4,2016,334
5,2017,334
6,2018,334
7,2019,352
8,2020,1387
9,2021,3345


## 6. Result and score inspection

For backtesting, we need completed matches only.

This checks whether result columns and score columns are present and usable.

In [10]:
result_cols = [col for col in ["FTR", "Res", "Result"] if col in df.columns]
score_cols = [col for col in ["FTHG", "FTAG", "HG", "AG"] if col in df.columns]

print("Candidate result columns:", result_cols)
print("Candidate score columns:", score_cols)

for col in result_cols:
    print(f"\nValue counts for {col}:")
    display(df[col].value_counts(dropna=False).reset_index(name="rows").rename(columns={"index": col}))

for col in score_cols:
    numeric = pd.to_numeric(df[col], errors="coerce")
    print(f"\nScore column: {col}")
    display(
        pd.DataFrame(
            {
                "metric": ["non_null", "numeric_parse_success", "min", "max", "mean"],
                "value": [
                    int(df[col].notna().sum()),
                    int(numeric.notna().sum()),
                    numeric.min(),
                    numeric.max(),
                    numeric.mean(),
                ],
            }
        )
    )

Candidate result columns: ['FTR', 'Res']
Candidate score columns: ['FTHG', 'FTAG', 'HG', 'AG']

Value counts for FTR:


,FTR,rows
0,H,6750
1,A,5014
2,NaN,4595
3,D,3937



Value counts for Res:


,Res,rows
0,NaN,15701
1,H,2052
2,A,1296
3,D,1247



Score column: FTHG


...

,metric,value
0,non_null,15701.000000
1,numeric_parse_success,15701.000000
2,min,0.000000
3,max,13.000000
4,mean,1.277307



Score column: HG


,metric,value
0,non_null,4595.000000
1,numeric_parse_success,4595.000000
2,min,0.000000
3,max,9.000000
4,mean,1.507073



Score column: AG


,metric,value
0,non_null,4595.000000
1,numeric_parse_success,4595.000000
2,min,0.000000
3,max,6.000000
4,mean,1.153428


In [11]:
if {"FTHG", "FTAG", "FTR"}.issubset(df.columns):
    score_home = pd.to_numeric(df["FTHG"], errors="coerce")
    score_away = pd.to_numeric(df["FTAG"], errors="coerce")

    derived_result = pd.Series(pd.NA, index=df.index, dtype="object")

    derived_result.loc[score_home > score_away] = "H"
    derived_result.loc[score_home == score_away] = "D"
    derived_result.loc[score_home < score_away] = "A"

    result_check = pd.DataFrame(
        {
            "raw_ftr": df["FTR"],
            "derived_ftr": derived_result,
        }
    )

    mismatch_mask = (
        result_check["raw_ftr"].notna()
        & result_check["derived_ftr"].notna()
        & (result_check["raw_ftr"] != result_check["derived_ftr"])
    )

    print(f"Rows with FTR/score mismatch: {int(mismatch_mask.sum()):,}")

    display(
        df.loc[
            mismatch_mask,
            ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"],
        ].head(20)
    )
else:
    print("Skipping FTR/score consistency check because FTHG, FTAG, or FTR is missing.")

Rows with FTR/score mismatch: 0


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR


## 7. Team column inspection

Day 3 should standardize home and away team names.  
For MVP, we probably avoid advanced team-name reconciliation unless we find obvious breakage.

In [12]:
team_cols = [col for col in ["HomeTeam", "AwayTeam", "Home", "Away"] if col in df.columns]
team_summary_records = []

for col in team_cols:
    team_summary_records.append(
        {
            "column": col,
            "non_null": int(df[col].notna().sum()),
            "null_count": int(df[col].isna().sum()),
            "unique_count": int(df[col].nunique(dropna=True)),
            "sample_values": df[col].dropna().astype(str).drop_duplicates().sort_values().head(20).tolist(),
        }
    )

pd.DataFrame(team_summary_records)

,column,non_null,null_count,unique_count,sample_values
0,HomeTeam,15701,4595,213,"[AVS, AZ Alkmaar, Ajaccio, Ajax, Alaves, Almere City, Almeria, Alverca, Anderlecht, Angers, Antwerp, Arouca, Arsenal..."
1,AwayTeam,15701,4595,213,"[AVS, AZ Alkmaar, Ajaccio, Ajax, Alaves, Almere City, Almeria, Alverca, Anderlecht, Angers, Antwerp, Arouca, Arsenal..."
2,Home,4595,15701,25,"[Atl. San Luis, Atlante, Atlas, Chiapas, Club America, Club Leon, Club Tijuana, Cruz Azul, Dorados de Sinaloa, Guada..."
3,Away,4595,15701,25,"[Atl. San Luis, Atlante, Atlas, Chiapas, Club America, Club Leon, Club Tijuana, Cruz Azul, Dorados de Sinaloa, Guada..."


In [13]:
if {"source_country", "HomeTeam", "AwayTeam"}.issubset(df.columns):
    team_counts = (
        df.groupby("source_country", dropna=False)
        .agg(
            home_teams=("HomeTeam", lambda s: s.nunique(dropna=True)),
            away_teams=("AwayTeam", lambda s: s.nunique(dropna=True)),
            rows=("HomeTeam", "size"),
        )
        .reset_index()
        .sort_values("source_country")
    )
    display(team_counts)
else:
    print("Skipping team count by source_country because required columns are missing.")

,source_country,home_teams,away_teams,rows
0,belgium,24,24,1782
1,england,28,28,2209
2,france,27,27,1994
3,germany,25,25,1773
4,italy,29,29,2200
5,mexico,0,0,4595
6,netherlands,26,26,1782
7,portugal,26,26,1771
8,spain,28,28,2190


## 8. Odds column inspection

For MVP we need one odds source for 1X2 markets:

```text
home win odds
draw odds
away win odds
```

Preferred starting point: `B365H`, `B365D`, `B365A` if coverage is good.

This section compares common bookmaker/average/maximum odds columns.

In [14]:
odds_triplets = {
    "Bet365": ["B365H", "B365D", "B365A"],
    "Pinnacle_or_PS": ["PSH", "PSD", "PSA"],
    "Average": ["AvgH", "AvgD", "AvgA"],
    "Maximum": ["MaxH", "MaxD", "MaxA"],
    "WilliamHill": ["WHH", "WHD", "WHA"],
    "Interwetten": ["IWH", "IWD", "IWA"],
}

odds_coverage_records = []

for source_name, cols in odds_triplets.items():
    existing = [col for col in cols if col in df.columns]
    all_exist = len(existing) == 3

    if all_exist:
        numeric = df[cols].apply(pd.to_numeric, errors="coerce")
        complete_rows = numeric.notna().all(axis=1)
        positive_rows = (numeric > 1).all(axis=1)
        valid_rows = complete_rows & positive_rows

        odds_coverage_records.append(
            {
                "odds_source": source_name,
                "home_col": cols[0],
                "draw_col": cols[1],
                "away_col": cols[2],
                "all_columns_exist": True,
                "complete_rows": int(complete_rows.sum()),
                "complete_pct": round(complete_rows.mean() * 100, 2),
                "valid_rows_gt_1": int(valid_rows.sum()),
                "valid_pct_gt_1": round(valid_rows.mean() * 100, 2),
                "min_home_odds": numeric[cols[0]].min(),
                "min_draw_odds": numeric[cols[1]].min(),
                "min_away_odds": numeric[cols[2]].min(),
            }
        )
    else:
        odds_coverage_records.append(
            {
                "odds_source": source_name,
                "home_col": cols[0],
                "draw_col": cols[1],
                "away_col": cols[2],
                "all_columns_exist": False,
                "complete_rows": None,
                "complete_pct": None,
                "valid_rows_gt_1": None,
                "valid_pct_gt_1": None,
                "min_home_odds": None,
                "min_draw_odds": None,
                "min_away_odds": None,
            }
        )

odds_coverage_df = pd.DataFrame(odds_coverage_records)
odds_coverage_df.sort_values(["all_columns_exist", "valid_rows_gt_1"], ascending=[False, False])

,odds_source,home_col,draw_col,away_col,all_columns_exist,complete_rows,complete_pct,valid_rows_gt_1,valid_pct_gt_1,min_home_odds,min_draw_odds,min_away_odds
2,Average,AvgH,AvgD,AvgA,True,15700,77.36,15700,77.36,1.04,2.09,1.08
3,Maximum,MaxH,MaxD,MaxA,True,15700,77.36,15700,77.36,1.06,2.20,1.10
0,Bet365,B365H,B365D,B365A,True,15690,77.31,15690,77.31,1.05,2.00,1.07
1,Pinnacle_or_PS,PSH,PSD,PSA,True,14864,73.24,14864,73.24,1.05,2.08,1.08
4,WilliamHill,WHH,WHD,WHA,True,12555,61.86,12555,61.86,1.03,2.20,1.05
5,Interwetten,IWH,IWD,IWA,True,9531,46.96,9531,46.96,1.04,2.10,1.08


In [15]:
if {"B365H", "B365D", "B365A"}.issubset(df.columns):
    b365 = df[["B365H", "B365D", "B365A"]].apply(pd.to_numeric, errors="coerce")
    b365_complete = b365.notna().all(axis=1)
    b365_valid = b365_complete & (b365 > 1).all(axis=1)

    print(f"B365 complete rows: {int(b365_complete.sum()):,} / {len(df):,}")
    print(f"B365 valid odds rows: {int(b365_valid.sum()):,} / {len(df):,}")

    display(
        df.loc[~b365_valid, ["source_country", "source_season", "source_league_code", "Date", "HomeTeam", "AwayTeam", "B365H", "B365D", "B365A"]]
        .head(30)
    )
else:
    print("B365 1X2 odds triplet not found.")

B365 complete rows: 15,690 / 20,296
B365 valid odds rows: 15,690 / 20,296


,source_country,source_season,source_league_code,Date,HomeTeam,AwayTeam,B365H,B365D,B365A
571,belgium,2122,B1,06/03/2022,Antwerp,Beerschot VA,NaN,NaN,NaN
1319,belgium,2425,B1,26/10/2024,Charleroi,Oud-Heverlee Leuven,NaN,NaN,NaN
1505,belgium,2425,B1,24/04/2025,Club Brugge,St. Gilloise,NaN,NaN,NaN
4017,france,2021,F1,13/09/2020,Paris SG,Marseille,NaN,NaN,NaN
4057,france,2021,F1,18/10/2020,Monaco,Montpellier,NaN,NaN,NaN
7793,italy,2021,I1,18/10/2020,Udinese,Parma,NaN,NaN,NaN
7795,italy,2021,I1,19/10/2020,Verona,Genoa,NaN,NaN,NaN
8341,italy,2122,I1,10/01/2022,Torino,Fiorentina,NaN,NaN,NaN
9958,mexico,all,MEX,21/07/2012,NaN,NaN,NaN,NaN,NaN
9959,mexico,all,MEX,21/07/2012,NaN,NaN,NaN,NaN,NaN


## 9. Market implied probability sanity check

This is not the Day 4 odds-processing module.  
This is only a rough inspection to see whether odds columns behave like decimal odds.

For decimal odds:

```text
implied probability = 1 / odds
bookmaker overround = sum(implied probabilities)
```

Overround should usually be above 1.0 for bookmaker odds.

In [16]:
if {"B365H", "B365D", "B365A"}.issubset(df.columns):
    b365 = df[["B365H", "B365D", "B365A"]].apply(pd.to_numeric, errors="coerce")
    valid = b365.notna().all(axis=1) & (b365 > 1).all(axis=1)
    implied = 1 / b365[valid]
    overround = implied.sum(axis=1)

    overround_summary = pd.DataFrame(
        {
            "metric": ["count", "min", "p01", "p05", "median", "mean", "p95", "p99", "max"],
            "value": [
                int(overround.count()),
                overround.min(),
                overround.quantile(0.01),
                overround.quantile(0.05),
                overround.median(),
                overround.mean(),
                overround.quantile(0.95),
                overround.quantile(0.99),
                overround.max(),
            ],
        }
    )
    display(overround_summary)

    suspicious = overround[(overround < 1.0) | (overround > 1.3)]
    print(f"Suspicious overround rows below 1.0 or above 1.3: {len(suspicious):,}")
else:
    print("Skipping implied probability inspection because B365 odds triplet is missing.")

,metric,value
0,count,15690.000000
1,min,1.027616
2,p01,1.041802
3,p05,1.046661
4,median,1.056322
5,mean,1.058733
6,p95,1.079745
7,p99,1.098485
8,max,1.166667


Suspicious overround rows below 1.0 or above 1.3: 0


## 10. Duplicate match-key inspection

Day 3 needs a stable match-level key.  
Likely candidate:

```text
source_country + source_league_code + Date + HomeTeam + AwayTeam
```

This checks whether that combination duplicates.

In [17]:
candidate_key = ["source_country", "source_league_code", "Date", "HomeTeam", "AwayTeam"]
missing_key_cols = [col for col in candidate_key if col not in df.columns]

if not missing_key_cols:
    duplicated_key_mask = df.duplicated(candidate_key, keep=False)
    print(f"Rows with duplicated candidate match key: {int(duplicated_key_mask.sum()):,}")

    if duplicated_key_mask.any():
        display(
            df.loc[duplicated_key_mask, candidate_key + ["source_season", "source_file"]]
            .sort_values(candidate_key)
            .head(50)
        )
else:
    print("Missing candidate key columns:", missing_key_cols)

Rows with duplicated candidate match key: 4,080


,source_country,source_league_code,Date,HomeTeam,AwayTeam,source_season,source_file
10495,mexico,MEX,01/02/2014,NaN,NaN,all,data\external\mexico\MEX.csv
10496,mexico,MEX,01/02/2014,NaN,NaN,all,data\external\mexico\MEX.csv
10497,mexico,MEX,01/02/2014,NaN,NaN,all,data\external\mexico\MEX.csv
10498,mexico,MEX,01/02/2014,NaN,NaN,all,data\external\mexico\MEX.csv
10824,mexico,MEX,01/02/2015,NaN,NaN,all,data\external\mexico\MEX.csv
10825,mexico,MEX,01/02/2015,NaN,NaN,all,data\external\mexico\MEX.csv
10826,mexico,MEX,01/02/2015,NaN,NaN,all,data\external\mexico\MEX.csv
10827,mexico,MEX,01/02/2015,NaN,NaN,all,data\external\mexico\MEX.csv
10828,mexico,MEX,01/02/2015,NaN,NaN,all,data\external\mexico\MEX.csv
12506,mexico,MEX,01/02/2020,NaN,NaN,all,data\external\mexico\MEX.csv


## 11. 2019+ eligibility inspection

For season-folder sources, Day 2 already selected 2019+ by folder.  
For country-level files like Mexico, Day 3 must filter rows using match date.

This section shows what date range each source country contributes.

In [18]:
if {"source_country", "Date"}.issubset(df.columns):
    parsed_date = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    date_coverage = (
        df.assign(_parsed_date=parsed_date)
        .groupby("source_country", dropna=False)
        .agg(
            rows=("source_country", "size"),
            parsed_dates=("_parsed_date", lambda s: int(s.notna().sum())),
            min_date=("_parsed_date", "min"),
            max_date=("_parsed_date", "max"),
            pre_2019_rows=("_parsed_date", lambda s: int((s < pd.Timestamp("2019-01-01")).sum())),
            rows_2019_plus=("_parsed_date", lambda s: int((s >= pd.Timestamp("2019-01-01")).sum())),
        )
        .reset_index()
        .sort_values("source_country")
    )
    display(date_coverage)
else:
    print("Skipping 2019+ eligibility inspection because source_country or Date is missing.")

C:\Users\jaymi\AppData\Local\Temp\ipykernel_29232\2477883142.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(_parsed_date=parsed_date)


,source_country,rows,parsed_dates,min_date,max_date,pre_2019_rows,rows_2019_plus
0,belgium,1782,1782,2020-08-08,2026-03-22,0,1782
1,england,2209,2209,2020-09-12,2026-03-22,0,2209
2,france,1994,1994,2020-08-21,2026-03-22,0,1994
3,germany,1773,1773,2020-09-18,2026-03-22,0,1773
4,italy,2200,2200,2020-09-19,2026-03-22,0,2200
5,mexico,4595,4595,2012-07-21,2026-03-23,2171,2424
6,netherlands,1782,1782,2020-09-12,2026-03-22,0,1782
7,portugal,1771,1771,2020-09-18,2026-03-22,0,1771
8,spain,2190,2190,2020-09-12,2026-03-22,0,2190


## 12. Suggested Day 3 system additions

Run all cells above first.  
Then use the outputs to fill this decision table.

This cell creates a starter recommendation table based on common football-data columns.

In [19]:
recommendations = []

def add_recommendation(item, decision, reason, file_path):
    recommendations.append(
        {
            "system_item": item,
            "suggested_decision": decision,
            "reason": reason,
            "likely_file_path": file_path,
        }
    )

# Column mapping recommendations
if "Date" in df.columns:
    add_recommendation(
        "Canonical date column",
        "Map raw Date -> match_date",
        "Date column exists and should be parsed with dayfirst=True.",
        "src/fbsystem/data/clean_matches.py",
    )

if {"HomeTeam", "AwayTeam"}.issubset(df.columns):
    add_recommendation(
        "Canonical team columns",
        "Map HomeTeam/AwayTeam -> home_team/away_team",
        "Common football-data team columns are present.",
        "src/fbsystem/data/clean_matches.py",
    )

if {"FTHG", "FTAG", "FTR"}.issubset(df.columns):
    add_recommendation(
        "Canonical result columns",
        "Map FTHG/FTAG/FTR -> home_goals/away_goals/full_time_result",
        "Full-time score and result are available for target construction.",
        "src/fbsystem/data/clean_matches.py",
    )

if {"B365H", "B365D", "B365A"}.issubset(df.columns):
    b365 = df[["B365H", "B365D", "B365A"]].apply(pd.to_numeric, errors="coerce")
    valid_pct = ((b365.notna().all(axis=1) & (b365 > 1).all(axis=1)).mean() * 100)
    add_recommendation(
        "Primary odds source",
        "Use B365H/B365D/B365A for MVP if coverage is acceptable",
        f"B365 triplet exists with about {valid_pct:.2f}% valid decimal odds rows.",
        "src/fbsystem/data/clean_matches.py",
    )

if "source_country" in df.columns:
    add_recommendation(
        "Source metadata retention",
        "Keep source_country/source_season/source_league_code/source_file in canonical data",
        "Needed for lineage, debugging, league-level evaluation, and walk-forward split checks.",
        "src/fbsystem/data/clean_matches.py",
    )

if {"source_country", "Date"}.issubset(df.columns):
    add_recommendation(
        "2019+ row filter",
        "Filter canonical data to match_date >= 2019-01-01",
        "Mexico is a country-level file, so file-level season filtering is not enough.",
        "src/fbsystem/data/clean_matches.py",
    )

add_recommendation(
    "Pandera schema",
    "Add a canonical match schema after cleaning",
    "Day 3 should validate required columns and basic value constraints.",
    "src/fbsystem/data/schemas.py",
)

add_recommendation(
    "Day 3 tests",
    "Add tests for date parsing, required columns, odds validity, result consistency, and 2019+ filtering",
    "Prevents quiet data bugs from becoming model bugs, which are harder and more embarrassing.",
    "tests/test_clean_matches.py",
)

recommendations_df = pd.DataFrame(recommendations)
recommendations_df

,system_item,suggested_decision,reason,likely_file_path
0,Canonical date column,Map raw Date -> match_date,Date column exists and should be parsed with dayfirst=True.,src/fbsystem/data/clean_matches.py
1,Canonical team columns,Map HomeTeam/AwayTeam -> home_team/away_team,Common football-data team columns are present.,src/fbsystem/data/clean_matches.py
2,Canonical result columns,Map FTHG/FTAG/FTR -> home_goals/away_goals/full_time_result,Full-time score and result are available for target construction.,src/fbsystem/data/clean_matches.py
3,Primary odds source,Use B365H/B365D/B365A for MVP if coverage is acceptable,B365 triplet exists with about 77.31% valid decimal odds rows.,src/fbsystem/data/clean_matches.py
4,Source metadata retention,Keep source_country/source_season/source_league_code/source_file in canonical data,"Needed for lineage, debugging, league-level evaluation, and walk-forward split checks.",src/fbsystem/data/clean_matches.py
5,2019+ row filter,Filter canonical data to match_date >= 2019-01-01,"Mexico is a country-level file, so file-level season filtering is not enough.",src/fbsystem/data/clean_matches.py
6,Pandera schema,Add a canonical match schema after cleaning,Day 3 should validate required columns and basic value constraints.,src/fbsystem/data/schemas.py
7,Day 3 tests,"Add tests for date parsing, required columns, odds validity, result consistency, and 2019+ filtering","Prevents quiet data bugs from becoming model bugs, which are harder and more embarrassing.",tests/test_clean_matches.py


## 13. Manual notes for Day 3

Use this section after running the notebook.

Recommended Day 3 outputs:

```text
data/processed/matches_canonical.parquet
outputs/evaluation/canonical_data_report.csv
```

Likely canonical columns:

```text
match_id
match_date
season
source_country
source_league_code
home_team
away_team
home_goals
away_goals
full_time_result
home_odds
draw_odds
away_odds
source_file
```

Likely Day 3 cleaning rules:

1. Parse `Date` using `dayfirst=True`.
2. Keep rows with `match_date >= 2019-01-01`.
3. Keep completed matches only.
4. Require non-null home/away teams.
5. Require valid result: `H`, `D`, or `A`.
6. Require valid numeric full-time goals.
7. Require decimal odds greater than 1.0.
8. Preserve source metadata.
9. Create a stable `match_id`.
10. Save canonical data as Parquet.

In [20]:
print("Raw data discovery complete.")
print("Review the recommendation table above before starting Day 3 production code.")

Raw data discovery complete.
Review the recommendation table above before starting Day 3 production code.
